<a href="https://colab.research.google.com/github/banteamlak1888/Floridan-University-Fundamentals-AI-Trianing/blob/main/Ethiopian_Customer_Request_Classification_(Transformers).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Transformer Models**
**Ethiopian Customer Request Classification Using Transformers**

**Step 1: Install Required Libraries**

First, we install the libraries needed for transformer fine-tuning.

In [ ]:
 # Install Hugging Face Transformers and PyTorch
!pip install -q transformers datasets evaluate accelerate scikit-learn sentencepiece

**Step 2: Upload Dataset**

In [ ]:
# Import file upload tool from Google Colab
from google.colab import files

# Upload the dataset CSV file
uploaded = files.upload()

Saving amharic_ethiopian_customer_request_large_dataset.csv to amharic_ethiopian_customer_request_large_dataset (1).csv


**Read Dataset**

In [ ]:
# Import pandas for data handling
import pandas as pd

# Read the uploaded CSV file
data = pd.read_csv("amharic_ethiopian_customer_request_large_dataset.csv")

# Show the first five rows
print(data.head())

                                                text   category
0  የመለያ መክፈቻ ሰነዶች ምንድናቸው በባህር ዳር። በፍጥነት እንዲፈታ እፈልጋለሁ    account
1                             የብድር ወለድ መጠን ምን ያህል ነው       loan
2                              ሰራተኛው በአክብሮት አላገለገለኝም  complaint
3  የመለያ ሚስጥር ቃል ማደስ እፈልጋለሁ በመቀሌ በሞባይል ባንኪንግ። መፍትሄ...    account
4                      የብድር ማመልከቻዬ ለምን ዘገየ። መረጃ ይስጡኝ       loan


**Step 3: Inspect Dataset**

In [ ]:
# Show dataset shape
print("Dataset shape:", data.shape)

# Show column names
print("Columns:", data.columns.tolist())

# Check missing values
print(data.isnull().sum())

# Show category distribution
print(data["category"].value_counts())

Dataset shape: (1650, 2)
Columns: ['text', 'category']
text        0
category    0
dtype: int64
category
technical issue    350
account            325
loan               325
complaint          325
money transfer     325
Name: count, dtype: int64


**Step 4: Encode Labels**

Transformer models need numerical labels, so we convert the request categories into numbers.

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

data["label"] = label_encoder.fit_transform(data["category"])

print(label_encoder.classes_)

data[["category", "label"]].head()

['account' 'complaint' 'loan' 'money transfer' 'technical issue']


,category,label
0,account,0
1,loan,2
2,complaint,1
3,account,0
4,loan,2


**Step 5: Split Dataset**

Now, we split the dataset into training and testing sets.

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    data,
    test_size=0.2,
    random_state=42,
    stratify=data["label"]
)

print(train_df.shape)
print(test_df.shape)

(1320, 3)
(330, 3)


**Step 6: Convert Pandas Data to Hugging Face Dataset**

The Hugging Face Trainer works best with Hugging Face Dataset format.

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df[["text", "label"]])

test_dataset = Dataset.from_pandas(test_df[["text", "label"]])

**Step 7: Load XLM-RoBERTa Tokenizer**

Now we load the tokenizer. The tokenizer converts Amharic text into tokens the transformer can understand.

In [ ]:
from transformers import AutoTokenizer

model_name = "xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

**Step 8: Tokenize Text**

Now we tokenize the Amharic customer requests.

In [ ]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_tokenized = train_dataset.map(tokenize_function, batched=True)

test_tokenized = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/1320 [00:00<?, ? examples/s]

Map:   0%|          | 0/330 [00:00<?, ? examples/s]

**Step 9: Load Transformer Model**

Now we load XLM-RoBERTa for sequence classification.

In [ ]:
from transformers import AutoModelForSequenceClassification

num_labels = len(label_encoder.classes_)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


**Step 10: Define Evaluation Metrics**

Now we define accuracy, precision, recall, and F1-score.

In [ ]:
import numpy as np

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted"
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

**Step 11: Define Training Arguments**

Now we define how the transformer model will be trained.

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./amharic_request_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_dir="./logs",
    logging_steps=20
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


**Step 12: Create Data Collator**

The data collator helps prepare batches during training.

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

**Step 13: Create Trainer**

Now we create the Hugging Face Trainer.

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

**Step 14: Train the Model**

Now we fine-tune the transformer model on our Amharic customer request dataset.

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.229038,0.039635,1.000000,1.000000,1.000000,1.000000
2,0.004860,0.001687,1.000000,1.000000,1.000000,1.000000
3,0.003048,0.001263,1.000000,1.000000,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=495, training_loss=0.33808999976726495, metrics={'train_runtime': 301.2868, 'train_samples_per_second': 13.144, 'train_steps_per_second': 1.643, 'total_flos': 260486961039360.0, 'train_loss': 0.33808999976726495, 'epoch': 3.0})

**Step 15: Evaluate the Model**

After training, we evaluate the model on unseen test data.

In [ ]:
results = trainer.evaluate()

print(results)

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.003048,0.001263,3,1.000000,1.000000,1.000000,1.000000


{'eval_loss': 0.0012629313860088587, 'eval_accuracy': 1.0, 'eval_precision': 1.0, 'eval_recall': 1.0, 'eval_f1': 1.0}


**Step 16: Make Predictions on Test Data**

Now we generate predictions from the trained model.

In [ ]:
predictions_output = trainer.predict(test_tokenized)

predicted_labels = np.argmax(
    predictions_output.predictions,
    axis=1
)

actual_labels = predictions_output.label_ids

**Step 17: Classification Report**

Now we generate a full classification report.

In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        actual_labels,
        predicted_labels,
        target_names=label_encoder.classes_
    )
)

**Step 18: Confusion Matrix (Counts)**

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    actual_labels,
    predicted_labels
)

print(cm)

**Confusion Matrix (Percentages)**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10,8))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)

plt.title(
    "Confusion Matrix - Amharic Customer Request Classification",
    fontsize=14
)

plt.xlabel(
    "Predicted Category",
    fontsize=12
)

plt.ylabel(
    "Actual Category",
    fontsize=12
)

plt.xticks(rotation=45)

plt.yticks(rotation=0)

plt.show()

**Step 19: Predict Sample Customer Requests from Dataset**

In [ ]:
sample_df = test_df[["text","category"]].head(10)

sample_df

**Step 20: Test A New Customer Request**

In [ ]:
sample_predictions = []

for text in sample_df["text"]:

    prediction = classify_request(text)

    sample_predictions.append(
        prediction
    )

In [ ]:
sample_df["Predicted Category"] = sample_predictions

sample_df

In [ ]:
comparison_df = sample_df.rename(
    columns={
        "category":"Actual Category"
    }
)

comparison_df

**Step 21: Save Model and Tokenizer**

Now we save the trained model and tokenizer. This is very important for deployment.

In [ ]:
model.save_pretrained("amharic_xlm_roberta_request_model")

tokenizer.save_pretrained("amharic_xlm_roberta_request_model")

**Step 22: Save Label Encoder**

We must also save the label encoder, because it converts model output numbers back into category names.

In [ ]:
import pickle

with open("label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

**Step 23: Load Saved Model Later**

Now we confirm that the saved model can be loaded again.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

loaded_tokenizer = AutoTokenizer.from_pretrained(
    "amharic_xlm_roberta_request_model"
)

loaded_model = AutoModelForSequenceClassification.from_pretrained(
    "amharic_xlm_roberta_request_model"
)

with open("label_encoder.pkl", "rb") as f:
    loaded_label_encoder = pickle.load(f)